# Snowpack -> Hydropower -> Electricity Prices

**Question:** Does winter/spring California snowpack predict summer electricity
price behavior through hydroelectric generation?

**Panel (data/processed/panel.csv):** one row per year --
`snowpack_pct` (April 1 snow water content as % of normal), summer day-ahead
price features (`price_mean`, `price_peak`, `price_vol`, `price_vol_hourly`),
and summer hydro generation (`hydro_gwh`, `hydro_gwh_eia`).

## Setup

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

## Mediation analysis: snowpack -> hydro output -> price volatility

In [2]:
from src.models import mediation_analysis, first_stage

panel = pd.read_csv(ROOT / 'data' / 'processed' / 'panel.csv', index_col=0)

# primary mediator: CAISO-reported summer hydro generation (no API key needed)
med = mediation_analysis(panel, target='price_vol', mediator='hydro_gwh')
med

{'n': 3,
 'total_effect_c': 0.11682904969162296,
 'p_total': 0.15945909348839052,
 'a_path': 11.598980438870552,
 'p_a': 0.2630388035627138,
 'b_mediator': 0.005878005063233805,
 'p_b': nan,
 'direct_effect_cp': 0.048650183943591864,
 'p_direct': nan,
 'proportion_mediated': 0.5835780221442626,
 'r2_joint': 1.0,
 'r2_total': 0.9385620311266106,
 'indirect_effect': 0.06817886574803096,
 'sobel_z': 0.0,
 'sobel_p': 1.0}

### Reading the paths (Baron-Kenny regression mediation)

In [3]:
import numpy as np
rows = {
    'total effect (c): snowpack -> volatility':  med['total_effect_c'],
    'a path: snowpack -> hydro':                 med['a_path'],
    'b path: hydro -> volatility (joint)':       med['b_mediator'],
    "direct effect (c'), snowpack | hydro":      med['direct_effect_cp'],
    "proportion mediated (1 - c'/c)":           med['proportion_mediated'],
}
for k, v in rows.items():
    print(f'{k:48s} {v:+.3f}')
print(f"\nSobel test of indirect effect: z={med['sobel_z']:.2f}, "
      f"p={med['sobel_p']:.3f} (n={med['n']})")

total effect (c): snowpack -> volatility         +0.117
a path: snowpack -> hydro                        +11.599
b path: hydro -> volatility (joint)              +0.006
direct effect (c'), snowpack | hydro             +0.049
proportion mediated (1 - c'/c)                   +0.584

Sobel test of indirect effect: z=0.00, p=1.000 (n=3)


**Causal-chain check:** if the snowpack effect runs *through* hydro,
then adding hydro to the regression should shrink the direct snowpack
coefficient (`c'` close to 0) while hydro keeps a significant `b` coefficient,
and the proportion mediated should be large.

> Mediation takeaway: fill in after running.